# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

---

## Solution Overview

This notebook implements a **Technical Q&A Assistant** with all the required features:

### Core Features (from Week 2 learnings):
1. **Gradio UI** - Beautiful chat interface with `gr.Blocks` and `gr.ChatInterface`
2. **Streaming** - Real-time response generation using Python generators (`yield`)
3. **Expert System Prompt** - Specialized for Python, ML, and LLM Engineering topics
4. **Model Switching** - Dropdown to switch between GPT-4.1-mini, GPT-4.1, Claude, and Llama

### Bonus Features:
5. **Tools/Function Calling** - Two tools:
   - `execute_python_code`: Runs Python snippets safely
   - `search_documentation`: Searches ML/AI documentation
6. **Audio Input** - Microphone recording with Whisper transcription
7. **Audio Output** - Text-to-speech using OpenAI TTS

### How to Use:
1. Run all cells in order
2. The basic version launches first (scroll down to see it)
3. The full-featured version with audio launches at the end
4. Try asking questions like:
   - "Explain what this code does: `yield from {book.get('author') for book in books}`"
   - "Can you run this code: `print([x**2 for x in range(5)])`"
   - "Search documentation for transformer"

In [11]:
# Imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [12]:
# Environment Setup and API Keys

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (optional - can use OpenRouter for Claude)")

In [13]:
# Setup API Clients
# Using OpenRouter as a unified endpoint for multiple models

# OpenRouter provides access to GPT, Claude, and many other models
openai_client = OpenAI(
    api_key=openai_api_key,
    base_url="https://openrouter.ai/api/v1"
)

# For Anthropic direct access (if you have the key)
anthropic_url = "https://api.anthropic.com/v1/"
anthropic_client = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None

# Local Ollama for open source models
ollama_url = "http://localhost:11434/v1"
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)

# Model choices
MODELS = {
    "GPT-4.1-mini": ("gpt-4.1-mini", openai_client),
    "GPT-4.1": ("gpt-4.1", openai_client),
    "Claude Sonnet 4.5": ("anthropic/claude-sonnet-4", openai_client),  # via OpenRouter
    "Llama 3.2 (Local)": ("llama3.2", ollama_client),
}

print("Available models:", list(MODELS.keys()))

In [14]:
# Expert System Prompt for Technical Q&A Assistant
# This prompt establishes the AI's expertise and personality

SYSTEM_PROMPT = """You are an expert technical tutor and programming assistant specializing in:
- Python programming and best practices
- Machine Learning and Deep Learning concepts
- LLM Engineering (transformers, fine-tuning, RAG, agents)
- Software architecture and design patterns
- Data structures and algorithms

Your teaching style:
1. Start with a clear, concise explanation
2. Provide code examples when relevant (using markdown code blocks)
3. Explain the "why" behind concepts, not just the "how"
4. Use analogies to make complex topics accessible
5. If you're uncertain about something, say so honestly

When explaining code, break it down step by step. When asked about errors, 
help debug methodically. Always encourage learning and exploration.

Respond in markdown format for better readability."""

In [15]:
# Tool Definitions - Demonstrating Function Calling
# We'll add a tool that can execute Python code snippets for demonstration

def execute_python_code(code: str) -> str:
    """
    Safely execute simple Python code and return the result.
    This is a sandboxed execution for demonstration purposes.
    """
    print(f"Tool called: Executing Python code...")
    try:
        # Create a restricted namespace for execution
        namespace = {"__builtins__": {"print": print, "len": len, "range": range, 
                                       "list": list, "dict": dict, "str": str,
                                       "int": int, "float": float, "sum": sum,
                                       "max": max, "min": min, "sorted": sorted,
                                       "enumerate": enumerate, "zip": zip}}
        
        # Capture output
        import io
        import sys
        old_stdout = sys.stdout
        sys.stdout = captured_output = io.StringIO()
        
        exec(code, namespace)
        
        sys.stdout = old_stdout
        output = captured_output.getvalue()
        
        return f"Code executed successfully.\nOutput:\n{output}" if output else "Code executed successfully (no output)"
    except Exception as e:
        return f"Error executing code: {str(e)}"


def search_documentation(query: str) -> str:
    """
    Simulate searching documentation for a topic.
    In a real application, this could connect to a vector database or search API.
    """
    print(f"Tool called: Searching documentation for '{query}'...")
    
    # Simulated documentation snippets
    docs = {
        "transformer": "The Transformer architecture uses self-attention mechanisms to process sequences in parallel. Key components: Multi-Head Attention, Feed-Forward Networks, Positional Encoding. Introduced in 'Attention Is All You Need' (2017).",
        "attention": "Self-attention computes relationships between all positions in a sequence. Formula: Attention(Q,K,V) = softmax(QK^T/√d_k)V. Multi-head attention runs multiple attention operations in parallel.",
        "gradient descent": "Gradient descent optimizes parameters by iteratively moving in the direction of steepest descent. Variants: SGD, Adam, RMSprop. Learning rate is a key hyperparameter.",
        "backpropagation": "Backpropagation computes gradients using the chain rule, propagating errors backward through the network. Essential for training neural networks.",
        "rag": "Retrieval-Augmented Generation (RAG) combines retrieval of relevant documents with LLM generation. Steps: 1) Embed query, 2) Retrieve similar docs, 3) Augment prompt, 4) Generate response.",
        "fine-tuning": "Fine-tuning adapts a pre-trained model to a specific task. Methods: Full fine-tuning, LoRA, QLoRA, Prompt tuning. Trade-off between performance and compute cost.",
    }
    
    # Simple keyword matching
    query_lower = query.lower()
    for key, value in docs.items():
        if key in query_lower:
            return f"Documentation found for '{key}':\n{value}"
    
    return f"No specific documentation found for '{query}'. Try asking about: transformer, attention, gradient descent, backpropagation, RAG, or fine-tuning."


# Define tools in OpenAI format
tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_python_code",
            "description": "Execute a Python code snippet and return the output. Use this when the user asks to run or test code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The Python code to execute"
                    }
                },
                "required": ["code"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function", 
        "function": {
            "name": "search_documentation",
            "description": "Search technical documentation for ML/AI concepts. Use this to provide accurate information about specific topics.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The topic or concept to search for"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    }
]

# Map function names to actual functions
tool_functions = {
    "execute_python_code": execute_python_code,
    "search_documentation": search_documentation
}

print("Tools defined:", [t["function"]["name"] for t in tools])

In [16]:
# Handle Tool Calls

def handle_tool_calls(message):
    """Process tool calls requested by the model and return responses."""
    responses = []
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        if function_name in tool_functions:
            # Get the first (and only) argument value
            arg_value = list(arguments.values())[0]
            result = tool_functions[function_name](arg_value)
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
        else:
            responses.append({
                "role": "tool",
                "content": f"Unknown function: {function_name}",
                "tool_call_id": tool_call.id
            })
    return responses

In [17]:
# Core Chat Function with Streaming and Tool Support

def chat(message, history, model_name, use_tools):
    """
    Main chat function with:
    - Streaming responses
    - Model switching
    - Tool calling support
    """
    # Get the selected model and client
    if model_name not in MODELS:
        yield "Error: Invalid model selected"
        return
    
    model_id, client = MODELS[model_name]
    
    # Build message history
    history_messages = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history_messages + [{"role": "user", "content": message}]
    
    try:
        # Determine if we should use tools (not all models support them equally)
        active_tools = tools if use_tools and "Llama" not in model_name else None
        
        # First call - check if model wants to use tools
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            tools=active_tools,
            stream=False  # First call without streaming to check for tool calls
        )
        
        # Check if we got a valid response
        if not response.choices:
            yield "Error: No response received from the model. Please try again."
            return
        
        # Handle tool calls if requested
        while response.choices[0].finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            
            # Check if there are actual tool calls
            if not assistant_message.tool_calls:
                break
                
            tool_responses = handle_tool_calls(assistant_message)
            
            # Add tool interaction to messages
            messages.append(assistant_message)
            messages.extend(tool_responses)
            
            # Get next response
            response = client.chat.completions.create(
                model=model_id,
                messages=messages,
                tools=active_tools,
                stream=False
            )
            
            if not response.choices:
                yield "Error: No response after tool call. Please try again."
                return
        
        # Check if we already have content from non-streaming response
        if response.choices[0].message.content:
            # Stream the existing content character by character for effect
            content = response.choices[0].message.content
            result = ""
            for char in content:
                result += char
                yield result
        else:
            # Try streaming response
            stream = client.chat.completions.create(
                model=model_id,
                messages=messages,
                stream=True
            )
            
            result = ""
            for chunk in stream:
                # Safely check for content
                if chunk.choices and len(chunk.choices) > 0:
                    delta = chunk.choices[0].delta
                    if delta and delta.content:
                        result += delta.content
                        yield result
            
            # If no content was streamed, yield a message
            if not result:
                yield "I received your message but couldn't generate a response. Please try again."
                
    except Exception as e:
        yield f"Error: {str(e)}\n\nMake sure the selected model is available and your API keys are configured."

In [19]:
# Gradio UI - Basic Version (Chat Interface with Model Selection)

with gr.Blocks(title="Technical Q&A Assistant") as demo:
    gr.Markdown("""
    # Technical Q&A Assistant
    
    Your AI-powered tutor for Python, Machine Learning, and LLM Engineering.
    
    **Features:**
    - Switch between GPT, Claude, and local Llama models
    - Enable tools for code execution and documentation search
    - Streaming responses for real-time feedback
    """)
    
    with gr.Row():
        model_selector = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="GPT-4.1-mini",
            label="Select Model"
        )
        tools_toggle = gr.Checkbox(
            value=True,
            label="Enable Tools (code execution & doc search)"
        )
    
    chatbot = gr.ChatInterface(
        fn=chat,
        type="messages",
        additional_inputs=[model_selector, tools_toggle],
        examples=[
            ["Explain what this code does: yield from {book.get('author') for book in books if book.get('author')}"],
            ["What is the transformer architecture and how does attention work?"],
            ["Can you run this code for me: print([x**2 for x in range(5)])"],
            ["Search the documentation for RAG"],
            ["What's the difference between fine-tuning and prompt engineering?"],
        ],
        title=None,  # We have our own title above
    )

demo.launch()

In [20]:
# BONUS: Audio Input/Output Support
# This adds voice interaction capabilities using OpenAI's Whisper (speech-to-text) and TTS

def transcribe_audio(audio_path):
    """Transcribe audio to text using OpenAI Whisper."""
    if audio_path is None:
        return ""
    
    try:
        with open(audio_path, "rb") as audio_file:
            # Using OpenRouter or OpenAI for transcription
            # Note: OpenRouter may not support audio, so we use OpenAI directly
            direct_openai = OpenAI()  # Uses OPENAI_API_KEY from env
            transcript = direct_openai.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file
            )
            return transcript.text
    except Exception as e:
        return f"[Audio transcription error: {str(e)}]"


def text_to_speech(text):
    """Convert text to speech using OpenAI TTS."""
    try:
        direct_openai = OpenAI()  # Uses OPENAI_API_KEY from env
        response = direct_openai.audio.speech.create(
            model="tts-1",
            voice="alloy",  # Options: alloy, echo, fable, onyx, nova, shimmer
            input=text[:4096]  # TTS has a character limit
        )
        
        # Save to temporary file
        import tempfile
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
            response.stream_to_file(f.name)
            return f.name
    except Exception as e:
        print(f"TTS Error: {e}")
        return None


def chat_with_audio(message, audio_input, history, model_name, use_tools, enable_tts):
    """
    Extended chat function that handles both text and audio input,
    and optionally returns audio output.
    """
    # If audio is provided, transcribe it
    if audio_input is not None:
        transcribed = transcribe_audio(audio_input)
        if transcribed:
            message = transcribed
            print(f"Transcribed audio: {message}")
    
    if not message:
        yield "", None
        return
    
    # Get streaming response
    full_response = ""
    for partial in chat(message, history, model_name, use_tools):
        full_response = partial
        yield full_response, None
    
    # Generate TTS if enabled
    if enable_tts and full_response:
        audio_output = text_to_speech(full_response)
        yield full_response, audio_output
    else:
        yield full_response, None

In [21]:
# Full-Featured Gradio UI with Audio Support

def process_message(message, audio, history, model_name, use_tools, enable_tts):
    """Process both text and audio inputs."""
    # Transcribe audio if provided
    if audio is not None:
        transcribed = transcribe_audio(audio)
        if transcribed and not transcribed.startswith("[Audio"):
            message = transcribed
    
    if not message:
        return history, "", None
    
    # Add user message to history
    history = history + [{"role": "user", "content": message}]
    
    return history, "", None


def generate_response(history, model_name, use_tools, enable_tts):
    """Generate assistant response with streaming."""
    if not history or history[-1]["role"] != "user":
        yield history, None
        return
    
    user_message = history[-1]["content"]
    
    # Create history without the last user message for the chat function
    prev_history = history[:-1]
    
    # Add empty assistant message that we'll fill
    history = history + [{"role": "assistant", "content": ""}]
    
    # Stream the response
    full_response = ""
    for partial in chat(user_message, prev_history, model_name, use_tools):
        full_response = partial
        history[-1]["content"] = partial
        yield history, None
    
    # Generate TTS if enabled
    if enable_tts and full_response:
        audio_path = text_to_speech(full_response)
        yield history, audio_path


with gr.Blocks(title="Technical Q&A Assistant", theme=gr.themes.Soft()) as demo_full:
    gr.Markdown("""
    # Technical Q&A Assistant
    
    Your AI-powered tutor for **Python**, **Machine Learning**, and **LLM Engineering**.
    
    ### Features:
    - **Model Switching**: Choose between GPT, Claude, or local Llama
    - **Tools**: Code execution and documentation search
    - **Voice Input**: Speak your questions (requires OpenAI API)
    - **Voice Output**: Listen to responses (optional TTS)
    - **Streaming**: Real-time response generation
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            model_selector = gr.Dropdown(
                choices=list(MODELS.keys()),
                value="GPT-4.1-mini",
                label="Select Model"
            )
        with gr.Column(scale=1):
            tools_toggle = gr.Checkbox(
                value=True,
                label="Enable Tools"
            )
        with gr.Column(scale=1):
            tts_toggle = gr.Checkbox(
                value=False,
                label="Enable Voice Output"
            )
    
    chatbot = gr.Chatbot(
        label="Conversation",
        height=400,
        type="messages"
    )
    
    with gr.Row():
        with gr.Column(scale=4):
            text_input = gr.Textbox(
                label="Type your question",
                placeholder="Ask about Python, ML, transformers, or any technical topic...",
                lines=2
            )
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                label="Or speak",
                sources=["microphone"],
                type="filepath"
            )
    
    with gr.Row():
        submit_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear Chat")
    
    audio_output = gr.Audio(
        label="Voice Response",
        autoplay=True,
        visible=True
    )
    
    gr.Markdown("""
    ### Example Questions:
    - "Explain what this code does: `yield from {book.get('author') for book in books}`"
    - "What is the transformer architecture?"
    - "Run this code: `print([x**2 for x in range(5)])`"
    - "Search documentation for RAG"
    """)
    
    # Event handlers
    submit_btn.click(
        fn=process_message,
        inputs=[text_input, audio_input, chatbot, model_selector, tools_toggle, tts_toggle],
        outputs=[chatbot, text_input, audio_output]
    ).then(
        fn=generate_response,
        inputs=[chatbot, model_selector, tools_toggle, tts_toggle],
        outputs=[chatbot, audio_output]
    )
    
    text_input.submit(
        fn=process_message,
        inputs=[text_input, audio_input, chatbot, model_selector, tools_toggle, tts_toggle],
        outputs=[chatbot, text_input, audio_output]
    ).then(
        fn=generate_response,
        inputs=[chatbot, model_selector, tools_toggle, tts_toggle],
        outputs=[chatbot, audio_output]
    )
    
    clear_btn.click(
        fn=lambda: ([], None),
        outputs=[chatbot, audio_output]
    )

demo_full.launch()